In [ ]:
import sys
import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Runtime setup successful.")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Pandas version: 2.2.3
NumPy version: 2.1.3
Runtime setup successful.


In [12]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path("/content/hrim_historical_training_baseline.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["structural_change_proxy"].value_counts())

Prepare baseline model

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

print("Scikit-learn and XGBoost imports successful.")

Train and evaluate the historical baseline

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/hrim_historical_training_baseline.csv")

# Six historical architecture features
feature_cols = [
    "previous_java_files",
    "previous_nonempty_lines",
    "previous_fan_in",
    "previous_fan_out",
    "previous_total_coupling",
    "previous_instability",
]

X = df[feature_cols]
y = df["structural_change_proxy"].astype(int)
groups = df["repository"]

logo = LeaveOneGroupOut()
results = []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    results.append({
        "repository": groups.iloc[test_idx].iloc[0],
        "test_examples": len(test_idx),
        "positive_examples": int(y_test.sum()),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "pr_auc": average_precision_score(y_test, probabilities),
        "roc_auc": (
            roc_auc_score(y_test, probabilities)
            if y_test.nunique() == 2 else np.nan
        ),
    })

results_df = pd.DataFrame(results)

display(results_df.round(4))

print("\nMean metrics across held-out repositories:")
display(
    results_df[
        ["precision", "recall", "f1", "pr_auc", "roc_auc"]
    ].mean().to_frame("mean").round(4)
)

In [ ]:
from google.colab import files

uploaded = files.upload()

Inspect fuse features

In [13]:
from google.colab import files

uploaded = files.upload()

Saving historical.zip to historical.zip


Extract the historical ZIP file

In [14]:
import zipfile
from pathlib import Path

zip_path = Path("/content/historical.zip")
extract_path = Path("/content/data/historical")

if not zip_path.exists():
    print("ZIP file not found. Please check the uploaded filename.")
else:
    extract_path.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    csv_files = list(extract_path.rglob("all_snapshots_architecture.csv"))

    print("Extraction completed.")
    print("Historical snapshot CSV files found:", len(csv_files))

    for file in csv_files:
        print(file)

Extraction completed.
Historical snapshot CSV files found: 11
/content/data/historical/historical/R09_kafca/all_snapshots_architecture.csv
/content/data/historical/historical/R03_jabref/all_snapshots_architecture.csv
/content/data/historical/historical/R02_common_lang/all_snapshots_architecture.csv
/content/data/historical/historical/R05_lucene/all_snapshots_architecture.csv
/content/data/historical/historical/R06_hadoop/all_snapshots_architecture.csv
/content/data/historical/historical/R11_key_cloak/all_snapshots_architecture.csv
/content/data/historical/historical/R04_argouml/all_snapshots_architecture.csv
/content/data/historical/historical/R01_spring_petclinic/all_snapshots_architecture.csv
/content/data/historical/historical/R08_apache_maven/all_snapshots_architecture.csv
/content/data/historical/historical/R10_teammates/all_snapshots_architecture.csv
/content/data/historical/historical/R07_junit/all_snapshots_architecture.csv


In [15]:
from pathlib import Path
import pandas as pd
import numpy as np

# Paths confirmed from your successful extraction
SNAPSHOT_ROOT = Path("/content/data/historical/historical")
OUTPUT_DIR = Path("/content/hrim_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

snapshot_files = list(
    SNAPSHOT_ROOT.rglob("all_snapshots_architecture.csv")
)

if len(snapshot_files) != 11:
    raise RuntimeError(
        f"Expected 11 snapshot files, found {len(snapshot_files)}"
    )

metrics = [
    "java_files",
    "nonempty_lines",
    "fan_in",
    "fan_out",
    "total_coupling",
    "instability",
]

frames = []

for file in snapshot_files:
    df = pd.read_csv(file)
    df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
    dates = sorted(df["snapshot_date"].unique())

    for previous_date, current_date in zip(dates[:-1], dates[1:]):
        previous = df[df["snapshot_date"] == previous_date]
        current = df[df["snapshot_date"] == current_date]

        pair = previous.merge(
            current,
            on=["repository", "package_path"],
            how="inner",
            suffixes=("_previous", "_current"),
            validate="one_to_one",
        )

        if pair.empty:
            continue

        changes = []
        for metric in metrics:
            before = pair[f"{metric}_previous"].astype(float)
            after = pair[f"{metric}_current"].astype(float)

            denominator = np.maximum(
                np.maximum(before.abs(), after.abs()), 1
            )
            changes.append((after - before).abs() / denominator)

        pair["mean_structural_change"] = pd.concat(
            changes, axis=1
        ).mean(axis=1)

        pair["structural_change_proxy"] = (
            pair["mean_structural_change"] >= 0.30
        ).astype(int)

        pair["previous_snapshot"] = previous_date.strftime("%Y-%m-%d")
        pair["current_snapshot"] = current_date.strftime("%Y-%m-%d")

        for metric in metrics:
            pair[f"previous_{metric}"] = pair[f"{metric}_previous"]

        columns = [
            "repository",
            "package_path",
            "previous_snapshot",
            "current_snapshot",
            "commit_hash_previous",
            "commit_hash_current",
            *[f"previous_{m}" for m in metrics],
            "mean_structural_change",
            "structural_change_proxy",
        ]

        frames.append(pair[columns])

transitions = pd.concat(frames, ignore_index=True)

output_path = OUTPUT_DIR / "hrim_aligned_architecture_transitions.csv"
transitions.to_csv(output_path, index=False)

print("SUCCESS — historical transition dataset created")
print("File:", output_path)
print("Rows:", len(transitions))
print("Repositories:", transitions["repository"].nunique())
print("Positive labels:", int(transitions["structural_change_proxy"].sum()))
print("Negative labels:", int((transitions["structural_change_proxy"] == 0).sum()))
print(
    "Duplicate transitions:",
    transitions.duplicated(
        ["repository", "package_path", "previous_snapshot", "current_snapshot"]
    ).sum(),
)

SUCCESS — historical transition dataset created
File: /content/hrim_outputs/hrim_aligned_architecture_transitions.csv
Rows: 3379
Repositories: 11
Positive labels: 520
Negative labels: 2859
Duplicate transitions: 0


Inspect the historical snapshot coverage

In [ ]:
from pathlib import Path
import pandas as pd

historical_root = Path("/content/data/historical/historical")

csv_files = sorted(
    historical_root.glob("*/all_snapshots_architecture.csv")
)

print("Repositories found:", len(csv_files))

for file in csv_files:
    df = pd.read_csv(file)

    print(
        f"{file.parent.name}: "
        f"{len(df)} rows, "
        f"{df['snapshot_date'].nunique()} snapshots, "
        f"{df['package_path'].nunique()} unique packages"
    )

    if file == csv_files[0]:
        print("\nColumn names:")
        print(df.columns.tolist())
        print("\nFirst five rows:")
        display(df.head())

Build historical HRIM features

In [ ]:
from pathlib import Path
import pandas as pd

historical_root = Path("/content/data/historical/historical")
csv_files = sorted(
    historical_root.glob("*/all_snapshots_architecture.csv")
)

frames = [pd.read_csv(file) for file in csv_files]
history = pd.concat(frames, ignore_index=True)

history["snapshot_date"] = pd.to_datetime(history["snapshot_date"])

print("Combined historical dataset:", history.shape)
print("Repositories:", history["repository"].nunique())
print("Snapshots:", history.groupby("repository")["snapshot_date"].nunique().sum())
print("Duplicate repository/date/package rows:",
      history.duplicated(
          ["repository", "snapshot_date", "package_path"]
      ).sum())
print("Missing values in required columns:",
      history[
          ["repository", "snapshot_date", "package_path",
           "java_files", "nonempty_lines", "fan_in",
           "fan_out", "total_coupling", "instability"]
      ].isna().sum().sum())

display(
    history.groupby("repository").agg(
        snapshots=("snapshot_date", "nunique"),
        package_snapshot_rows=("package_path", "size"),
        unique_packages=("package_path", "nunique")
    ).reset_index()
)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Load all historical repository snapshots
historical_root = Path("/content/data/historical/historical")

frames = [
    pd.read_csv(file)
    for file in sorted(
        historical_root.glob("*/all_snapshots_architecture.csv")
    )
]

history = pd.concat(frames, ignore_index=True)
history["snapshot_date"] = pd.to_datetime(history["snapshot_date"])

metrics = [
    "java_files",
    "nonempty_lines",
    "fan_in",
    "fan_out",
    "total_coupling",
    "instability",
]

# Sort snapshots and package records chronologically
history = history.sort_values(
    ["repository", "package_path", "snapshot_date"]
).reset_index(drop=True)

# Find the previous observation of each package
grouped = history.groupby(["repository", "package_path"], sort=False)

for col in metrics:
    history[f"previous_{col}"] = grouped[col].shift(1)

history["previous_snapshot"] = grouped["snapshot_date"].shift(1)
history["previous_commit_hash"] = grouped["commit_hash"].shift(1)

# Keep only package observations present in consecutive snapshots.
# A gap means this is not necessarily an adjacent-snapshot transition.
history["previous_snapshot_count"] = grouped.cumcount()
history["current_snapshot_index"] = (
    history.groupby(["repository", "snapshot_date"])["snapshot_date"]
    .transform(lambda x: 0)
)

# Check which repositories have packages in every snapshot
coverage = (
    history.groupby(["repository", "snapshot_date"])["package_path"]
    .count()
    .reset_index(name="package_rows")
)

print("Snapshot package counts:")
display(coverage)

print(
    "\nRows with a previous observation:",
    int(history["previous_snapshot"].notna().sum())
)

Build  aligned historical transitions

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

historical_root = Path("/content/data/historical/historical")

frames = [
    pd.read_csv(file)
    for file in sorted(
        historical_root.glob("*/all_snapshots_architecture.csv")
    )
]

history = pd.concat(frames, ignore_index=True)
history["snapshot_date"] = pd.to_datetime(history["snapshot_date"])

metrics = [
    "java_files",
    "nonempty_lines",
    "fan_in",
    "fan_out",
    "total_coupling",
    "instability",
]

transition_frames = []

for repository, repo_df in history.groupby("repository"):
    dates = sorted(repo_df["snapshot_date"].unique())

    for i in range(1, len(dates)):
        previous_date = pd.Timestamp(dates[i - 1])
        current_date = pd.Timestamp(dates[i])

        previous = repo_df[
            repo_df["snapshot_date"] == previous_date
        ][["package_path"] + metrics].copy()

        current = repo_df[
            repo_df["snapshot_date"] == current_date
        ][["package_path"] + metrics].copy()

        # Only compare packages present in both adjacent snapshots
        matched = previous.merge(
            current,
            on="package_path",
            how="inner",
            suffixes=("_previous", "_current"),
            validate="one_to_one",
        )

        if matched.empty:
            continue

        matched["repository"] = repository
        matched["previous_snapshot"] = previous_date
        matched["current_snapshot"] = current_date

        # Calculate the symmetric relative change for each metric
        changes = []

        for metric in metrics:
            old = matched[f"{metric}_previous"].astype(float)
            new = matched[f"{metric}_current"].astype(float)

            denominator = old.abs() + new.abs()

            relative_change = np.where(
                denominator == 0,
                0.0,
                (new - old).abs() / denominator,
            )

            matched[f"change_{metric}"] = relative_change
            changes.append(relative_change)

        matched["mean_structural_change"] = np.column_stack(
            changes
        ).mean(axis=1)

        matched["structural_change_proxy"] = (
            matched["mean_structural_change"] >= 0.30
        ).astype(int)

        # Predictors use only the previous snapshot
        for metric in metrics:
            matched[f"previous_{metric}"] = matched[
                f"{metric}_previous"
            ]

        transition_frames.append(matched)

transitions = pd.concat(transition_frames, ignore_index=True)

print("Matched package transitions:", len(transitions))
print("Repositories:", transitions["repository"].nunique())
print(
    "Positive proxy labels:",
    int(transitions["structural_change_proxy"].sum())
)
print(
    "Positive-label rate:",
    round(transitions["structural_change_proxy"].mean(), 4)
)

display(
    transitions.groupby("repository").agg(
        transitions=("package_path", "size"),
        positive_labels=("structural_change_proxy", "sum"),
        mean_change=("mean_structural_change", "mean"),
    ).reset_index()
)

In [ ]:
from pathlib import Path

output_dir = Path("/content/hrim_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "hrim_historical_transitions.csv"

transitions.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(transitions))
print("Columns:", len(transitions.columns))
print("File size (KB):", round(output_path.stat().st_size / 1024, 2))

Train and evaluate the historical baseline

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

# Load the saved historical transitions
df = pd.read_csv("/content/hrim_outputs/hrim_historical_transitions.csv")

# Use only previous-snapshot metrics as predictors
features = [
    "previous_java_files",
    "previous_nonempty_lines",
    "previous_fan_in",
    "previous_fan_out",
    "previous_total_coupling",
    "previous_instability",
]

X = df[features].astype(float)
y = df["structural_change_proxy"].astype(int)
groups = df["repository"]

logo = LeaveOneGroupOut()
results = []
all_predictions = []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    test_repo = groups.iloc[test_idx].iloc[0]

    # Handle the class imbalance using the training fold only
    positives = int(y_train.sum())
    negatives = len(y_train) - positives
    scale_weight = negatives / max(positives, 1)

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    # ROC-AUC is undefined if the test set contains only one class
    roc_auc = (
        roc_auc_score(y_test, probabilities)
        if y_test.nunique() == 2
        else np.nan
    )

    row = {
        "repository": test_repo,
        "test_samples": len(test_idx),
        "positive_labels": int(y_test.sum()),
        "precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "pr_auc": average_precision_score(y_test, probabilities),
        "roc_auc": roc_auc,
    }

    results.append(row)

    all_predictions.append(pd.DataFrame({
        "repository": test_repo,
        "actual": y_test.to_numpy(),
        "probability": probabilities,
        "predicted": predictions,
    }))

results_df = pd.DataFrame(results)
predictions_df = pd.concat(all_predictions, ignore_index=True)

print("Per-repository results:")
display(results_df.round(4))

print("\nMean metrics across repositories:")
display(
    results_df[
        ["precision", "recall", "f1", "pr_auc", "roc_auc"]
    ].mean().to_frame("mean").round(4)
)

results_df.to_csv(
    "/content/hrim_outputs/hrim_loro_baseline_metrics.csv",
    index=False,
)

predictions_df.to_csv(
    "/content/hrim_outputs/hrim_loro_baseline_predictions.csv",
    index=False,
)

print("\nSaved metrics and predictions in /content/hrim_outputs/")

Calculate the pooled evaluation metrics

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# Use the out-of-repository predictions already created
pred = predictions_df.copy()

y_true = pred["actual"]
y_pred = pred["predicted"]
y_prob = pred["probability"]

print("Pooled evaluation metrics")
print("-------------------------")
print(f"Test samples: {len(pred)}")
print(f"Actual positive labels: {int(y_true.sum())}")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"PR-AUC:    {average_precision_score(y_true, y_prob):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_true, y_prob):.4f}")

print("\nConfusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification report:")
print(classification_report(y_true, y_pred, zero_division=0))

establish the baseline comparison

In [ ]:
import json
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)

output_dir = Path("/content/hrim_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

y_true = predictions_df["actual"]
y_pred = predictions_df["predicted"]
y_prob = predictions_df["probability"]

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

report = {
    "model": "Historical six-metric XGBoost baseline",
    "validation": "Leave-one-repository-out",
    "target": "structural_change_proxy",
    "test_samples": int(len(y_true)),
    "positive_labels": int(y_true.sum()),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    "pr_auc": float(average_precision_score(y_true, y_prob)),
    "roc_auc": float(roc_auc_score(y_true, y_prob)),
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
    "features": [
        "previous_java_files",
        "previous_nonempty_lines",
        "previous_fan_in",
        "previous_fan_out",
        "previous_total_coupling",
        "previous_instability",
    ],
}

report_path = output_dir / "hrim_historical_baseline_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nSaved report to:", report_path)

In [16]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score
)
from xgboost import XGBClassifier

DATA = Path("/content/hrim_outputs/hrim_aligned_architecture_transitions.csv")
df = pd.read_csv(DATA)

print("Dataset:", df.shape)
print("Columns:", df.columns.tolist())
print("Target distribution:")
print(df["structural_change_proxy"].value_counts())

# Use only features measured at the previous snapshot.
feature_candidates = [
    "previous_java_files",
    "previous_nonempty_lines",
    "previous_fan_in",
    "previous_fan_out",
    "previous_total_coupling",
    "previous_instability",
]

# Stop clearly if the dataset uses different column names.
missing = [c for c in feature_candidates if c not in df.columns]
if missing:
    print("\nMissing expected feature columns:", missing)
    print("Please share the column list printed above.")
else:
    X = df[feature_candidates].apply(
        pd.to_numeric, errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

    y = df["structural_change_proxy"].astype(int)
    groups = df["repository"]

    logo = LeaveOneGroupOut()
    all_y, all_pred, all_prob = [], [], []

    for train_idx, test_idx in logo.split(X, y, groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Impute using the training fold only.
        medians = X_train.median().fillna(0)
        X_train = X_train.fillna(medians)
        X_test = X_test.fillna(medians)

        model = XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=max(
                1, (y_train == 0).sum() / max(1, (y_train == 1).sum())
            ),
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        probabilities = model.predict_proba(X_test)[:, 1]
        predictions = (probabilities >= 0.5).astype(int)

        all_y.extend(y_test.tolist())
        all_pred.extend(predictions.tolist())
        all_prob.extend(probabilities.tolist())

    print("\n=== Historical baseline: Leave-One-Repository-Out ===")
    print("Precision:", round(precision_score(all_y, all_pred, zero_division=0), 4))
    print("Recall:   ", round(recall_score(all_y, all_pred, zero_division=0), 4))
    print("F1-score: ", round(f1_score(all_y, all_pred, zero_division=0), 4))
    print("PR-AUC:   ", round(average_precision_score(all_y, all_prob), 4))
    print("ROC-AUC:  ", round(roc_auc_score(all_y, all_prob), 4))

    output = Path("/content/hrim_outputs/hrim_baseline_loro_metrics.csv")
    pd.DataFrame([{
        "model": "XGBoost historical architecture baseline",
        "validation": "Leave-One-Repository-Out",
        "precision": precision_score(all_y, all_pred, zero_division=0),
        "recall": recall_score(all_y, all_pred, zero_division=0),
        "f1": f1_score(all_y, all_pred, zero_division=0),
        "pr_auc": average_precision_score(all_y, all_prob),
        "roc_auc": roc_auc_score(all_y, all_prob),
    }]).to_csv(output, index=False)

    print("\nMetrics saved to:", output)


Dataset: (3379, 14)
Columns: ['repository', 'package_path', 'previous_snapshot', 'current_snapshot', 'commit_hash_previous', 'commit_hash_current', 'previous_java_files', 'previous_nonempty_lines', 'previous_fan_in', 'previous_fan_out', 'previous_total_coupling', 'previous_instability', 'mean_structural_change', 'structural_change_proxy']
Target distribution:
structural_change_proxy
0    2859
1     520
Name: count, dtype: int64

=== Historical baseline: Leave-One-Repository-Out ===
Precision: 0.2061
Recall:    0.4135
F1-score:  0.2751
PR-AUC:    0.2093
ROC-AUC:   0.5769

Metrics saved to: /content/hrim_outputs/hrim_baseline_loro_metrics.csv


In [17]:

import pandas as pd
from pathlib import Path

# Existing files — reuse, don't recreate
OUT = Path("/content/hrim_outputs")
TRANSITIONS = OUT / "hrim_aligned_architecture_transitions.csv"

FEATURE_FILES = {
    "architecture_evolution_graph": Path("/content/hrim_fused_features.csv"),
    "semantic_fusion": Path("/content/hrim_semantic_fused_features.csv"),
    "dependency_graph": Path("/content/dependency_graph_features.csv"),
}

# Load the historical dataset you already created
transitions = pd.read_csv(TRANSITIONS)

print("Historical transitions:", transitions.shape)
print("Repositories:", transitions["repository"].nunique())

# Inspect feature files and their repository/package keys
for name, path in FEATURE_FILES.items():
    print(f"\n--- {name} ---")

    if not path.exists():
        print("File missing:", path)
        continue

    features = pd.read_csv(path)
    print("Shape:", features.shape)

    key_columns = [
        c for c in ["repository", "package_path"]
        if c in features.columns
    ]
    print("Available join keys:", key_columns)

    if len(key_columns) == 2:
        feature_keys = features[key_columns].drop_duplicates()
        transition_keys = transitions[
            ["repository", "package_path"]
        ].drop_duplicates()

        matched = transition_keys.merge(
            feature_keys,
            on=["repository", "package_path"],
            how="inner"
        )

        print("Unique historical package keys:", len(transition_keys))
        print("Matched feature keys:", len(matched))
        print("Key coverage:", round(
            len(matched) / max(1, len(transition_keys)) * 100, 2
        ), "%")
    else:
        print("Cannot calculate package-key coverage: key columns missing.")

    for col in [
        "semantic_features_available",
        "graph_features_available",
    ]:
        if col in features.columns:
            print(f"{col} counts:")
            print(features[col].value_counts(dropna=False).to_dict())

print("\nCheck complete. No files were modified.")


Historical transitions: (3379, 14)
Repositories: 11

--- architecture_evolution_graph ---
Shape: (2707, 24)
Available join keys: ['repository', 'package_path']
Unique historical package keys: 2060
Matched feature keys: 1864
Key coverage: 90.49 %
graph_features_available counts:
{1: 2648, 0: 59}

--- semantic_fusion ---
Shape: (2707, 793)
Available join keys: ['repository', 'package_path']
Unique historical package keys: 2060
Matched feature keys: 1864
Key coverage: 90.49 %
semantic_features_available counts:
{False: 1827, True: 880}
graph_features_available counts:
{1: 2648, 0: 59}

--- dependency_graph ---
Shape: (2749, 8)
Available join keys: ['repository', 'package_path']
Unique historical package keys: 2060
Matched feature keys: 1912
Key coverage: 92.82 %

Check complete. No files were modified.


In [18]:

import pandas as pd
from pathlib import Path

OUT = Path("/content/hrim_outputs")

transitions = pd.read_csv(
    OUT / "hrim_aligned_architecture_transitions.csv"
)

fused = pd.read_csv("/content/hrim_fused_features.csv")
semantic = pd.read_csv("/content/hrim_semantic_fused_features.csv")
dependency = pd.read_csv("/content/dependency_graph_features.csv")

KEYS = ["repository", "package_path"]

# Prevent duplicate feature keys from multiplying transition rows.
for name, frame in [
    ("fused", fused),
    ("semantic", semantic),
    ("dependency", dependency),
]:
    duplicates = frame.duplicated(KEYS).sum()
    print(f"{name}: duplicate package keys = {duplicates}")

    if duplicates:
        raise ValueError(
            f"{name} contains duplicate repository/package keys. "
            "Resolve them before joining."
        )

# Prefix feature names to prevent accidental column-name collisions.
def prefix_features(frame, prefix):
    rename = {
        c: f"{prefix}_{c}"
        for c in frame.columns
        if c not in KEYS
    }
    return frame.rename(columns=rename)

fused_p = prefix_features(fused, "fused")
semantic_p = prefix_features(semantic, "semantic")
dependency_p = prefix_features(dependency, "dep")

# Left joins preserve all historical transitions.
fusion = transitions.merge(
    fused_p, on=KEYS, how="left", validate="many_to_one"
)
fusion = fusion.merge(
    semantic_p, on=KEYS, how="left", validate="many_to_one"
)
fusion = fusion.merge(
    dependency_p, on=KEYS, how="left", validate="many_to_one"
)

print("\nFusion table shape:", fusion.shape)
print("Historical transitions retained:", len(fusion))
print("Repositories:", fusion["repository"].nunique())

for col in [
    "fused_semantic_features_available",
    "semantic_semantic_features_available",
    "fused_graph_features_available",
    "dep_graph_features_available",
]:
    if col in fusion.columns:
        print(f"{col}:")
        print(fusion[col].value_counts(dropna=False).to_dict())

output = OUT / "hrim_historical_fusion_join_audit.csv"
fusion.to_csv(output, index=False)

print("\nSaved joined audit dataset:", output)
print("Note: package-key joins do not establish snapshot-time alignment.")


fused: duplicate package keys = 0
semantic: duplicate package keys = 0
dependency: duplicate package keys = 0

Fusion table shape: (3379, 833)
Historical transitions retained: 3379
Repositories: 11
semantic_semantic_features_available:
{False: 2213, True: 928, nan: 238}
fused_graph_features_available:
{1.0: 3105, nan: 238, 0.0: 36}

Saved joined audit dataset: /content/hrim_outputs/hrim_historical_fusion_join_audit.csv
Note: package-key joins do not establish snapshot-time alignment.


In [19]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score
)
from xgboost import XGBClassifier

OUT = Path("/content/hrim_outputs")
DATA = OUT / "hrim_historical_fusion_join_audit.csv"
df = pd.read_csv(DATA)

TARGET = "structural_change_proxy"
KEYS = {
    "repository", "package_path",
    "previous_snapshot", "current_snapshot",
    "commit_hash_previous", "commit_hash_current",
    "mean_structural_change", TARGET,
}

# Select the existing fused and dependency features.
# Exclude semantic embeddings and metadata/availability flags.
feature_cols = [
    c for c in df.columns
    if (
        c.startswith("fused_") or c.startswith("dep_")
    )
    and not c.startswith("fused_semantic_emb_")
    and not c.endswith("_features_available")
    and c not in KEYS
]

# Retain numeric features only.
X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.loc[:, X.notna().any(axis=0)]

y = df[TARGET].astype(int)
groups = df["repository"]

print("Rows:", len(df))
print("Selected features:", X.shape[1])
print("Target counts:", y.value_counts().to_dict())
print("Feature examples:", X.columns[:15].tolist())

logo = LeaveOneGroupOut()
all_y, all_pred, all_prob = [], [], []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Fit imputation on training repositories only.
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=max(
            1, (y_train == 0).sum() / max(1, (y_train == 1).sum())
        ),
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    all_y.extend(y_test.tolist())
    all_pred.extend(predictions.tolist())
    all_prob.extend(probabilities.tolist())

metrics = {
    "model": "Exploratory architecture-evolution-dependency fusion",
    "validation": "Leave-One-Repository-Out",
    "rows": len(df),
    "features": X.shape[1],
    "precision": precision_score(all_y, all_pred, zero_division=0),
    "recall": recall_score(all_y, all_pred, zero_division=0),
    "f1": f1_score(all_y, all_pred, zero_division=0),
    "pr_auc": average_precision_score(all_y, all_prob),
    "roc_auc": roc_auc_score(all_y, all_prob),
}

print("\n=== Exploratory fusion results ===")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

pd.DataFrame([metrics]).to_csv(
    OUT / "hrim_exploratory_fusion_metrics.csv", index=False
)
print("\nSaved:", OUT / "hrim_exploratory_fusion_metrics.csv")


Rows: 3379
Selected features: 24
Target counts: {0: 2859, 1: 520}
Feature examples: ['fused_java_files', 'fused_lines_of_code', 'fused_fan_in', 'fused_fan_out', 'fused_total_coupling', 'fused_instability', 'fused_commit_count', 'fused_contributor_count', 'fused_files_changed', 'fused_total_insertions', 'fused_total_deletions', 'fused_total_churn', 'fused_graph_in_degree', 'fused_graph_out_degree', 'fused_graph_total_degree']

=== Exploratory fusion results ===
model: Exploratory architecture-evolution-dependency fusion
validation: Leave-One-Repository-Out
rows: 3379
features: 24
precision: 0.2717
recall: 0.4269
f1: 0.3321
pr_auc: 0.2662
roc_auc: 0.6641

Saved: /content/hrim_outputs/hrim_exploratory_fusion_metrics.csv


In [20]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score
)
from xgboost import XGBClassifier

OUT = Path("/content/hrim_outputs")
df = pd.read_csv(OUT / "hrim_historical_fusion_join_audit.csv")

TARGET = "structural_change_proxy"
KEYS = {
    "repository", "package_path", "previous_snapshot",
    "current_snapshot", "commit_hash_previous",
    "commit_hash_current", "mean_structural_change", TARGET,
}

# Full HRIM: existing fusion features + GraphCodeBERT embeddings
features = [
    c for c in df.columns
    if (
        c.startswith("fused_") or
        c.startswith("dep_") or
        c.startswith("semantic_semantic_emb_")
    )
    and not c.endswith("_features_available")
    and c not in KEYS
]

X = df[features].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.loc[:, X.notna().any(axis=0)]

y = df[TARGET].astype(int)
groups = df["repository"]

print("Rows:", len(df))
print("Features:", X.shape[1])
print("Semantic embedding columns:",
      sum(c.startswith("semantic_semantic_emb_") for c in X.columns))

logo = LeaveOneGroupOut()
all_y, all_pred, all_prob = [], [], []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=max(
            1, (y_train == 0).sum() / max(1, (y_train == 1).sum())
        ),
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)

    all_y.extend(y_test.tolist())
    all_pred.extend(pred.tolist())
    all_prob.extend(prob.tolist())

results = {
    "model": "Full HRIM with GraphCodeBERT (exploratory)",
    "validation": "Leave-One-Repository-Out",
    "rows": len(df),
    "features": X.shape[1],
    "precision": precision_score(all_y, all_pred, zero_division=0),
    "recall": recall_score(all_y, all_pred, zero_division=0),
    "f1": f1_score(all_y, all_pred, zero_division=0),
    "pr_auc": average_precision_score(all_y, all_prob),
    "roc_auc": roc_auc_score(all_y, all_prob),
}

print("\n=== Full HRIM results (exploratory) ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

pd.DataFrame([results]).to_csv(
    OUT / "hrim_full_semantic_fusion_metrics.csv", index=False
)
print("\nSaved full HRIM metrics.")


Rows: 3379
Features: 792
Semantic embedding columns: 768

=== Full HRIM results (exploratory) ===
model: Full HRIM with GraphCodeBERT (exploratory)
validation: Leave-One-Repository-Out
rows: 3379
features: 792
precision: 0.2450
recall: 0.3519
f1: 0.2889
pr_auc: 0.2319
roc_auc: 0.6238

Saved full HRIM metrics.


In [21]:

from pathlib import Path
import zipfile
from google.colab import files

OUT = Path("/content/hrim_outputs")
ZIP_PATH = Path("/content/hrim_pp1_backup.zip")

# Collect all files generated in the HRIM output folder.
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for path in OUT.rglob("*"):
        if path.is_file():
            z.write(path, arcname=str(path.relative_to(OUT)))

print("Backup created:", ZIP_PATH)
print("Files saved:", len(list(OUT.rglob("*"))))
files.download(str(ZIP_PATH))


Backup created: /content/hrim_pp1_backup.zip
Files saved: 6


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>